# Tutorial 20 — Work with Hugging Face models

Tutorial 10 verified the local QAChat environment. This tutorial introduces the
model files that QAChat can load from the standard Hugging Face cache.

You will inspect cached models, resolve the backend selected by QAChat, choose
the corresponding canonical QAChat chat-model default, check whether that model
is already available locally, and optionally download it. The notebook does not
load the model into memory yet; [Tutorial 30](./30_model_chat.ipynb) does that.

Each tutorial is independently runnable, so you do not need to execute
[Tutorial 10](./10_setup.ipynb) first.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
from huggingface_hub.constants import HF_HUB_CACHE

print("Hugging Face cache:", HF_HUB_CACHE)


In [ ]:
import qachat


## 2. Inspect the Hugging Face cache

Hugging Face uses a shared cache outside the QAChat repository. The models shown
here may have been downloaded by QAChat or by another application.

A cached model is not automatically suitable for every task or backend. MLX
normally requires MLX-formatted weights, while the PyTorch backend uses
Transformers-compatible weights. A cache entry may also be an embedding, image,
or other non-chat model.


In [ ]:
models = qachat.available_models()
if models:
    print("Model IDs found in the Hugging Face cache:\n")
    for number, model in enumerate(models, start=1):
        size_gb = model["size_bytes"] / 1_000_000_000
        print(f"{number}. {model['model_id']} ({size_gb:.2f} GB)")
else:
    print("No models are cached yet.")


## 3. Choose a backend-compatible default model

Rather than selecting the first arbitrary cache entry, resolve QAChat's backend
and choose the canonical default chat model for that backend. These constants
are defaults, not an allowlist: you can replace `MODEL_ID` with another
compatible Hugging Face model ID when needed.


In [ ]:
REQUESTED_BACKEND = qachat.BACKEND_AUTO

SELECTED_BACKEND = qachat.choose_backend(REQUESTED_BACKEND)

MODEL_ID = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if SELECTED_BACKEND == "mlx"
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)


In [ ]:
print(f"Requested backend: {REQUESTED_BACKEND}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Selected model: {MODEL_ID}")


In [ ]:
if qachat.model_is_available(MODEL_ID):
    print(f"{MODEL_ID} is already cached")
else:
    print(f"{MODEL_ID} is NOT cached")


## 4. Download only when requested

`model_is_available()` checks whether the selected model already has a usable
cached snapshot. Keep `DOWNLOAD = False` if you only want to inspect the cache.
Set it to `True` to download missing files into the normal Hugging Face cache.
Existing correct files are reused, and an interrupted download can be resumed by
rerunning the cell.


In [ ]:
DOWNLOAD = False


In [ ]:
if DOWNLOAD:
    saved_path = qachat.download_model(MODEL_ID)
    print("Cached snapshot:", saved_path)
else:
    print("Download skipped.")


The Hugging Face cache remains outside this Git repository by default. Model
weights are runtime data and should not be copied into the QAChat source tree or
committed to Git.


## Continue the tutorial series

**Previous:** [Tutorial 10 — Check the QAChat setup](./10_setup.ipynb)

**Next:** [Tutorial 30 — Load a model and chat](./30_model_chat.ipynb)
